# 02. 비용 인식 하네스 정책 비교

목표: 무호출, 항상 호출, 불확실할 때만 호출하는 정책을 같은 장난감 과제에서 비교합니다. 성공률만 높이는 정책과 성공·비용을 함께 최적화하는 정책이 다를 수 있음을 확인합니다.

In [ ]:
from dataclasses import dataclass
from statistics import mean

@dataclass(frozen=True)
class Scenario:
    name: str
    uncertainty: float
    base_success: float

scenarios = [
    Scenario('명확한 단일 목표', 0.10, 0.95),
    Scenario('두 위치 중 탐색', 0.45, 0.62),
    Scenario('오래된 위치 기억', 0.80, 0.28),
    Scenario('세 단계 계획', 0.65, 0.42),
]

def no_harness(s):
    return 0

def always_harness(s):
    return 3  # track, commit, recall을 매번 사용

def selective_harness(s):
    # 불확실성이 큰 과제만 Progress와 Experience를 확인합니다.
    return 2 if s.uncertainty >= 0.5 else 0


## 성공과 비용을 함께 계산

하네스 호출은 불확실성에 비례해 성공 가능성을 올리지만 호출당 비용이 듭니다. 세 번을 넘기면 스팸 벌점도 적용합니다. 실제 논문 보상의 정확한 복제가 아니라 설계 직관을 분리해 보는 모형입니다.

In [ ]:
def evaluate(policy, call_cost=0.35):
    rows = []
    for scenario in scenarios:
        calls = policy(scenario)
        # 첫 두 호출의 가치가 크고 이후 수익은 체감한다고 가정합니다.
        useful_calls = min(calls, 2)
        success_prob = min(0.99, scenario.base_success + 0.38 * scenario.uncertainty * useful_calls)
        efficiency = 1.0 - 0.08 * calls
        spam_penalty = max(0, calls - 2) * 0.25
        reward = 10 * success_prob + efficiency - call_cost * calls - spam_penalty
        rows.append((scenario.name, success_prob, calls, reward))
    return rows

policies = {
    '무호출': no_harness,
    '항상 호출': always_harness,
    '선택 호출': selective_harness,
}

summary = {}
for name, policy in policies.items():
    rows = evaluate(policy)
    summary[name] = {
        '평균 성공확률': mean(row[1] for row in rows),
        '평균 호출수': mean(row[2] for row in rows),
        '평균 보상': mean(row[3] for row in rows),
    }

for name, metrics in summary.items():
    print(name, {k: round(v, 3) for k, v in metrics.items()})


## 호출 비용 민감도

호출 비용이 바뀌면 최적 정책도 달라집니다. 운영 환경에서는 토큰 가격만이 아니라 지연, 도구 실패, 잘못된 기억을 읽는 위험도 비용에 포함해야 합니다.

In [ ]:
for cost in (0.0, 0.25, 0.5, 1.0):
    scores = {
        name: mean(row[3] for row in evaluate(policy, call_cost=cost))
        for name, policy in policies.items()
    }
    winner = max(scores, key=scores.get)
    print(f'호출 비용={cost:.2f} -> {winner}, 보상={scores[winner]:.3f}')

assert summary['선택 호출']['평균 호출수'] < summary['항상 호출']['평균 호출수']
assert summary['선택 호출']['평균 성공확률'] > summary['무호출']['평균 성공확률']
print('선택 정책은 호출을 줄이면서 무호출보다 높은 성공 가능성을 유지합니다.')
